# ola

In [16]:
#Importar bibliotecas
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from scipy.sparse import hstack, csr_matrix

# Fazer download dos pacotes necessários do NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Configura o pandas para não esconder colunas
pd.set_option('display.max_columns', None)

[nltk_data] Downloading package punkt to C:\Users\Big
[nltk_data]     Boss\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Big
[nltk_data]     Boss\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Big
[nltk_data]     Boss\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
# Conta a quantidade de palavras em maiusculas
def count_caps_letters(text):
    # Proteção contra valores nulos/NaNs
    if pd.isna(text) or not isinstance(text, str):
        return 0
    return sum(1 for char in text if char.isupper())

def clean_text(text):
        #mantém apenas letras (a-z, A-Z), espaços e números, remove o resto. Coloca tudo em minusculas
        text = re.sub(r'[^\w\s]', '', text.lower()) 
        return text

def remove_stopwords_and_lemmatize(tokens):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    # Remove stopwords e aplica lematização
    return [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

In [3]:
#Ler ambos csvs
df_true = pd.read_csv('data/True.csv')
df_false = pd.read_csv('data/Fake.csv')

#classe -> se 0, então a notícia é verdadeira; se 1, então a notícia é fake
#vamos adicionar coluna classe
df_true['classe'] = 0
df_false['classe'] = 1

#vamos juntar ambas databases :)
df_total = pd.concat([df_true, df_false], ignore_index=True)
print(df_total.info())

#randomizing a database para as linhas com diferentes classes estarem mistruradas
df_total = df_total.sample(frac=1).reset_index(drop=True)
print(df_total.head())

# export dataframe to csv
df_total.to_csv('data/total.csv', index=False)

<class 'pandas.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    44898 non-null  str  
 1   text     44898 non-null  str  
 2   subject  44898 non-null  str  
 3   date     44898 non-null  str  
 4   classe   44898 non-null  int64
dtypes: int64(1), str(4)
memory usage: 1.7 MB
None
                                               title  \
0  WATCH: RACIST CONGRESSWOMAN MAXINE WATERS Won’...   
1  Don't count on Germany's SPD to rescue Macron'...   
2  Obama announces lifting of U.S. sanctions on M...   
3  Pentagon drops Maine from list of potential mi...   
4  GAY MARRIAGE APPROVED BY SUPREME COURT With Ir...   

                                                text       subject  \
0  Close your eyes and picture Speaker of the Hou...      politics   
1  BERLIN (Reuters) - During Germany s election c...     worldnews   
2  WASHINGTON (Reuters) - President Barack Obama ... 

In [4]:
# Conta todos os "!" no texto e no título
df_total['count_exclamation_title'] = df_total['title'].apply(lambda x: x.count('!'))
df_total['count_exclamation_text'] = df_total['text'].apply(lambda x: x.count('!'))
    
# Conta a quantidade de palavras em maiusculas no título e no texto
df_total['count_caps_letters_title'] = df_total['title'].apply(count_caps_letters)
df_total['count_caps_letters_text'] = df_total['text'].apply(count_caps_letters)

In [8]:
print("\n--- A processar os Títulos ---")
# Limpeza básica (remover pontuação e minúsculas)
df_total['title_clean'] = df_total['title'].apply(clean_text)
# Tokenização
df_total['tokens_title'] = df_total['title_clean'].apply(word_tokenize)
# Stopwords e Lematização
df_total['tokens_title_clean'] = df_total['tokens_title'].apply(remove_stopwords_and_lemmatize)
# Juntar os tokens numa string final para o TF-IDF
df_total['final_title'] = df_total['tokens_title_clean'].apply(lambda x: ' '.join(x))

print("\n--- A processar os Textos ---")
# Limpeza básica (remover pontuação e minúsculas)
df_total['text_clean'] = df_total['text'].apply(clean_text)
# Tokenização
df_total['tokens_text'] = df_total['text_clean'].apply(word_tokenize)
# Stopwords e Lematização
df_total['tokens_text_clean'] = df_total['tokens_text'].apply(remove_stopwords_and_lemmatize)
# Juntar os tokens numa string final para o TF-IDF
df_total['final_text'] = df_total['tokens_text_clean'].apply(lambda x: ' '.join(x))


--- A processar os Títulos ---

--- A processar os Textos ---


In [ ]:
# QUESTAO, ter duas matrizes diferentes para o tf idf, uma para o título e outra para o texto? ou juntar os dois e fazer só uma matriz?

# Aplicar TF-IDF ao título e ao texto
# tfidf_title = TfidfVectorizer(max_features=1000)
# X_title = tfidf_title.fit_transform(df_total['final_title'])

# tfidf_text = TfidfVectorizer(max_features=4000)
# X_text = tfidf_text.fit_transform(df_total['final_text'])

# X_final = hstack([X_title, X_text])

# print(X_final.shape)
    
# print("Pré-processamento e matrizes concluídos com sucesso!")

(44898, 5000)
Pré-processamento e matrizes concluídos com sucesso!


In [ ]:
# X = X_final
# y = df_total['classe']
    
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
# 1. Isolar as características (X) e a variável alvo (y)
# Vamos incluir os textos pré-processados e as métricas extra que criaste
X = df_total[[
    'final_title', 'final_text', 
    'count_exclamation_title', 'count_exclamation_text', 
    'count_caps_letters_title', 'count_caps_letters_text'
]]
y = df_total['classe']

# 2. FAZER A DIVISÃO PRIMEIRO (Prevenção de Data Leakage)
X_train_df, X_test_df, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Configurar os vetores TF-IDF
tfidf_title = TfidfVectorizer(max_features=1000)
tfidf_text = TfidfVectorizer(max_features=4000)

# 4. APLICAR TF-IDF AOS DADOS DE TREINO (Aprender o vocabulário: fit_transform)
X_train_title_tfidf = tfidf_title.fit_transform(X_train_df['final_title'])
X_train_text_tfidf = tfidf_text.fit_transform(X_train_df['final_text'])

# 5. APLICAR TF-IDF AOS DADOS DE TESTE (Apenas aplicar o que foi aprendido: transform)
X_test_title_tfidf = tfidf_title.transform(X_test_df['final_title'])
X_test_text_tfidf = tfidf_text.transform(X_test_df['final_text'])

# 6. Preparar as variáveis numéricas extra (Maiúsculas e Exclamações)
# Convertê-las para matrizes esparsas para serem compatíveis com as matrizes do TF-IDF
extra_features_train = csr_matrix(X_train_df[[
    'count_exclamation_title', 'count_exclamation_text', 
    'count_caps_letters_title', 'count_caps_letters_text'
]].values)

extra_features_test = csr_matrix(X_test_df[[
    'count_exclamation_title', 'count_exclamation_text', 
    'count_caps_letters_title', 'count_caps_letters_text'
]].values)

# 7. Juntar tudo (TF-IDF do título + TF-IDF do texto + Variáveis extra)
X_train_final = hstack([X_train_title_tfidf, X_train_text_tfidf, extra_features_train])
X_test_final = hstack([X_test_title_tfidf, X_test_text_tfidf, extra_features_test])

print(f"Dimensões de X_train_final: {X_train_final.shape}")
print(f"Dimensões de X_test_final: {X_test_final.shape}")
print("Pré-processamento e matrizes concluídos com sucesso e sem data leakage!")

Dimensões de X_train_final: (35918, 5004)
Dimensões de X_test_final: (8980, 5004)
Pré-processamento e matrizes concluídos com sucesso e sem data leakage!
